# Supplementary Figure 9

Predicted metagenome (MG) database size and FastaLake peptide gain in CapScan, with peptides of 9-50 residues. Panel
a plots, for each of 250 matched specimens, the median gain of FastaLake over the specimen-matched MG search against
the number of predicted-protein records in that MG database. Panel b gives the median acquisition-level gain in thirds
of MG database size over 226 intestinal-fluid acquisitions. The input table is a parameter (`panels.ACQ_9_50_TSV`,
environment `SFIG09_ACQ_TSV`) because the CapScan MG benchmark is being extended to 295 acquisitions.

Run from the repository root: `python paper.py build SFig09`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

import os
import pandas as pd
FONT_DIR = os.environ.get("FASTALAKE_FONT_DIR") or os.environ.get("FL_STYLE_DIR")
t = panels.load(SD)

## Panel a

Specimen-level FastaLake gain over the MG search (9-50 residues) against predicted-protein records in the
specimen-matched MG database (log axis; gain axis linear to 50% and logarithmic above; lower limit lowered to show the
one specimen at -0.03%). Supports the legend (v30f line 238, v28d line 445): "Each point is one of 250 matched
specimens." n = 250 specimens (282 acquisitions; repeated acquisitions summarised by their median). No test.

The 9-50 per-acquisition table is first rebuilt from the 282-acquisition table and the recount and compared with the
shipped copy; then the v30 figure is drawn (both panels on one canvas), with the producer drawing run alongside and
every plotted value compared.

In [2]:
assert panels.check_producer_asserts(t)
acq9_rebuilt = panels.acquisitions_9_50(t)
shipped = pd.read_csv(SD / "capscan_acquisitions_9_50.tsv", sep="\t")
print("9-50 table rebuilt from source_data equals shipped copy:",
      acq9_rebuilt.reset_index(drop=True).equals(shipped.reset_index(drop=True)))
res = panels.render_v30(SD, OUT, UNIT, FONT_DIR)
print("input is the v30 table:", res["is_v29_input"], "; panel a y minimum:", round(res["ymin"], 2))
print("specimens:", panels.stat_a_n_specimens(res["spec"]))
from IPython.display import Image
Image(url="../../build/SFig09/SupplementaryFigure9.png", embed=False, width=720)   # linked, not embedded

9-50 table rebuilt from source_data equals shipped copy: True


  [style] SupplementaryFigure9: 0 label(s) raised to 5 pt, 2 capped to 7 pt, 0 tick axis/axes clamped, 0 line(s) raised to 1 pt


input is the v30 table: True ; panel a y minimum: -1.07
specimens: 250


## Panel b

Median acquisition-level gain in increasing thirds of intestinal-fluid MG database size (equal-count bins). Supports
the legend (v30f line 238): "70.4%, 58.3% and 42.0% (76, 75 and 75 acquisitions)", and Results (v30f line 71):
"same-spectrum FastaLake gains were 70%, 58% and 42% across increasing thirds of intestinal-fluid metagenome database
size". n = 226 acquisitions (76, 75, 75) from 14 participants; acquisitions are repeated within specimens and
participants. Descriptive medians, no test and no interval. Drawn in the render above.

In [3]:
tb = panels.stat_b_tertile_medians(res["caps"])
print(tb.to_string(index=False))

     bin  median_gain  n  min_size  max_size
Smallest    70.439735 76      4957     80944
  Middle    58.333333 75     81639    141039
 Largest    41.960947 75    142612    403423


## Checks and numbers

Pixel checks of the v30 composite (PNG at 300 dpi, PDF rasterised at 300 dpi) against the published v30 render; they
need the Söhne fonts (`FASTALAKE_FONT_DIR`). Then every number in `NUMBERS.tsv` is printed in table order.

In [4]:
from tools.check_pixels import compare
if not fl._HAVE_STYLE:
    print("pixel checks SKIPPED: fonts absent")
else:
    for name, new, exp, kw in [("SFig09_v30", res["png"], "SFig09_v30.png", {}),
                               ("SFig09_v30_pdf", res["pdf"], "SFig09_v30.pdf", {"dpi": 300})]:
        r = compare(new, UNIT / "expected" / exp, UNIT / "checks" / name, **kw)
        print(name, r["verdict"], r.get("differing_pixels"), r.get("max_channel_difference"))
print(sum(c["passed"] for c in res["checks"]), "of", len(res["checks"]), "value checks passed")

acq9 = res["acq"]
db = panels.stat_129_db_size(acq9)
wp = panels.stat_129_within_participant(acq9)
dna = panels.stat_129_dna(acq9)
y = panels.stat_129_pooled_yield(acq9)
med = tb.set_index("bin")
rows = [
    ("445", "250 matched specimens", panels.stat_a_n_specimens(res["spec"])),
    ("445", "linear up to 50%", panels.LINTHRESH),
    ("445", "logarithmic above 50%", panels.LINTHRESH),
    ("445", "70.4%", med.loc["Smallest", "median_gain"]),
    ("445", "58.3%", med.loc["Middle", "median_gain"]),
    ("445", "42.0%", med.loc["Largest", "median_gain"]),
    ("445", "76 acquisitions", med.loc["Smallest", "n"]),
    ("445", "75 acquisitions", med.loc["Middle", "n"]),
    ("445", "75 acquisitions", med.loc["Largest", "n"]),
    ("129", "54 ng (lowest fluid region)", dna["fluid_region_min"]),
    ("129", "168 ng (highest fluid region)", dna["fluid_region_max"]),
    ("129", "687 ng (stool)", dna["stool"]),
    ("129", "99 acquisitions with DNA", dna["n_with_dna"]),
    ("129", "107,000 (fluid DB median)", db["fluid_median"]),
    ("129", "272,000 (stool DB median)", db["stool_median"]),
    ("129", "All 14 participants with both", wp["n_both"]),
    ("129", "14 each smaller", wp["n_fluid_smaller"]),
    ("129", "34% (MG yield)", y["mg_percent"]),
    ("129", "53% (FastaLake yield)", y["fl_percent"]),
    ("129", "70%", med.loc["Smallest", "median_gain"]),
    ("129", "58%", med.loc["Middle", "median_gain"]),
    ("129", "42%", med.loc["Largest", "median_gain"]),
]
for line, what, value in rows:
    value = value.item() if hasattr(value, "item") else value
    print(f"{line}\t{what}\t{value}")
print("participants:", wp["n_participants"], "stool only:", wp["stool_only"])
print("DNA medians by region:", dna["by_region"])

SFig09_v30 IDENTICAL 0 0


SFig09_v30_pdf IDENTICAL 0 0
24 of 24 value checks passed
445	250 matched specimens	250
445	linear up to 50%	50
445	logarithmic above 50%	50
445	70.4%	70.43973452923825
445	58.3%	58.33333333333333
445	42.0%	41.960947237224765
445	76 acquisitions	76
445	75 acquisitions	75
445	75 acquisitions	75
129	54 ng (lowest fluid region)	54.39906774723238
129	168 ng (highest fluid region)	168.41464060596428
129	687 ng (stool)	686.9272736903438
129	99 acquisitions with DNA	99
129	107,000 (fluid DB median)	106512.5
129	272,000 (stool DB median)	272332.0
129	All 14 participants with both	14
129	14 each smaller	14
129	34% (MG yield)	34.389169066293334
129	53% (FastaLake yield)	52.65844666225505
129	70%	70.43973452923825
129	58%	58.33333333333333
129	42%	41.960947237224765
participants: 15 stool only: [15]
DNA medians by region: {'colon': 168.41464060596428, 'duodenum': 54.39906774723238, 'ileum': 112.37353673393719, 'jejunum': 117.69691191270724, 'stool': 686.9272736903438}
